# Experiment 1 — Static FA/GI CART preprocessing

Inspect the exact patient-level matrix **before model fitting**. This notebook imports the baseline script and calls its `prepare_inputs()` function, so mean aggregation, FA/GI labels, and the recorded patient split use the same implementation.

It does not train CART, change the dataset, or overwrite the experiment results. Run all cells with a Python kernel that has the existing project dependencies; CART is not required for this preview. Open the notebook from this experiment folder or from the repository root.

The baseline now uses **one mean per clinical feature (15 columns)**. The script trains FA and GI separately and saves train/test AUCs, a combined test ROC, separate learned-tree figures, and reproducibility metadata. Historical GI results from the previous aggregation are kept separately in `previous_mean_min_max_results`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the repository whether Jupyter starts here or at the repository root.
relative_folder = Path("experiments") / "experiment_1_xgboost_baseline"
repository_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / relative_folder / "experiment1_cart_baseline.py").is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError("Start Jupyter inside TemporalDecisionTree_clean.")
experiment_folder = repository_root / relative_folder
for folder in (repository_root, experiment_folder):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import experiment1_cart_baseline as baseline

print("Baseline implementation:", baseline.__file__)

## Original longitudinal data

The saved Experiment 1 cohort has many observations per patient. The raw preview includes `time` and the anchor rows for context; **these are excluded from the model matrix**. FA and GI labels are loaded separately and never enter the clinical feature matrix. Both outcomes reuse the same recorded patient split.

In [ ]:
longitudinal, source_labels = baseline.experiment.load_inputs()
print(f"Longitudinal rows: {len(longitudinal):,}")
print(f"Patients: {longitudinal['patient_id'].nunique():,}")
display(longitudinal.head(10))
display(
    longitudinal.groupby("feature_name").agg(
        observations=("value", "size"),
        missing_values=("value", lambda values: values.isna().sum()),
    )
)

## Run the identical preprocessing

For every clinical feature, use exactly **one full-timeline mean**. Numeric features use the arithmetic mean; binary features use the proportion of nonmissing observations that are positive. There are 15 feature columns, all ending in `__mean`; no `__min` or `__max` features are produced. `ClinicalVisitYN__mean` is retained to keep one column per clinical feature, although it is constant and provides no discrimination.

Missing observations are skipped when calculating a mean; an all-missing or absent feature stays `NaN`. There is no imputation, scaling, or feature selection, and each patient's mean uses only their own values.

There are no categorical strings in this dataset. Timestamps, anchors, windows, offsets, observation counts, IDs, and labels are absent from the feature columns. Patient IDs remain only as the DataFrame index for alignment.

The recorded split takes priority over the current source seed: the saved run used seed 42, while the current Temporal Tree source specifies 48.

In [ ]:
X_train, X_test, targets = baseline.prepare_inputs()
assert set(targets) == {"fa", "gi"}
X = pd.concat([X_train, X_test]).sort_index()

shape_summary = pd.DataFrame({
    "partition": ["all", "train", "test"],
    "patients": [len(X), len(X_train), len(X_test)],
    "features": [X.shape[1], X_train.shape[1], X_test.shape[1]],
})
display(shape_summary)

## What enters CART

`X_train` below is precisely the training feature matrix passed to `model.fit()`. Every row is one patient and every column is a static clinical summary. The transposed preview makes all feature names and values easier to read. Labels are kept in `targets`, outside this matrix.

In [ ]:
display(X_train.head(8))
display(X_train.head(3).T.rename_axis("static_feature"))

## Trace two features back to raw observations

For one training patient, compare the full-timeline means of `LengthCm` (numeric measurement) and `VomitingYN` (binary indicator) against their raw values. Every clinical feature has exactly one retained mean column. Dates and ordering are not used.

In [ ]:
patient_id = X_train.index[0]
print("Example training patient:", patient_id)
for feature in ("LengthCm", "VomitingYN"):
    values = pd.to_numeric(
        longitudinal.loc[
            longitudinal["patient_id"].eq(patient_id)
            & longitudinal["feature_name"].eq(feature),
            "value",
        ]
    )
    print(f"\n{feature}: raw values (original stored order, unused by aggregation)")
    display(pd.DataFrame({"value": values.to_numpy()}))
    column = f"{feature}__mean"
    raw_mean = values.mean()
    model_value = X_train.loc[patient_id, column]
    np.testing.assert_allclose(model_value, raw_mean, equal_nan=True)
    display(pd.DataFrame([{
        "feature": feature,
        "aggregation": "mean",
        "raw_mean": raw_mean,
        "model_value": model_value,
    }]))

## Training distributions and missing values

These tables describe the training inputs. Missing measurements stay missing for CART to handle; no held-out values are used to fill them.

In [ ]:
display(X_train.describe().T)
missing = pd.DataFrame({
    "missing_train": X_train.isna().sum(),
    "missing_fraction_train": X_train.isna().mean(),
    "dtype": X_train.dtypes.astype(str),
}).rename_axis("static_feature")
display(missing)

## Labels and alignment — display only

This preview combines labels and partition information for inspection. It is **not** a model input. FA and GI labels are shown for alignment inspection only. The original patient split is reused unchanged.

In [ ]:
alignment_preview = pd.DataFrame(index=X.index)
alignment_preview["partition"] = np.where(
    X.index.isin(X_train.index), "train", "test"
)
for outcome in ("fa", "gi"):
    alignment_preview[f"{outcome.upper()}_label"] = targets[outcome].reindex(X.index)
display(alignment_preview.head(10))
display(
    alignment_preview.groupby("partition").agg(
        patients=("GI_label", "size"),
        FA_positive_prevalence=("FA_label", "mean"),
        GI_positive_prevalence=("GI_label", "mean"),
    )
)

## Validate the matrices

Check patient isolation, feature names/types, and label alignment. If a recorded split exists, verify membership and labels directly against it. The baseline loader also checks all three input hashes against the recorded experiment configuration.

Full-timeline summaries are appropriate for this specified benchmark. Excluding explicit timestamps does not remove possible age associations in clinical measurements; this notebook does not establish an early-prediction cutoff.

In [ ]:
assert set(targets) == {"fa", "gi"}
assert X.index.is_unique
assert X.shape[1] == len(baseline.NUMERIC_FEATURES | baseline.BINARY_FEATURES)
assert set(X.columns) == {
    f"{feature}__mean"
    for feature in baseline.NUMERIC_FEATURES | baseline.BINARY_FEATURES
}
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X.index) == set(longitudinal["patient_id"])
assert X_train.columns.equals(X_test.columns)
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in X.dtypes)
assert all(
    column.split("__")[0] in baseline.NUMERIC_FEATURES | baseline.BINARY_FEATURES
    and column.split("__")[1] == "mean"
    for column in X.columns
)
for outcome in ("fa", "gi"):
    original = source_labels[outcome].set_index("patient_id")["label"]
    pd.testing.assert_series_equal(targets[outcome], original.reindex(X.index))
    for frame in (X_train, X_test):
        assert targets[outcome].loc[frame.index].index.equals(frame.index)
        assert targets[outcome].loc[frame.index].nunique() == 2

split_path = baseline.experiment.OUTPUT_DIR / "shared_patient_split.csv"
if split_path.exists():
    saved = pd.read_csv(split_path).set_index("patient_id")
    assert set(X_train.index) == set(saved.index[saved["split"].eq("train")])
    assert set(X_test.index) == set(saved.index[saved["split"].eq("test")])
    for outcome in ("fa", "gi"):
        np.testing.assert_array_equal(
            targets[outcome].loc[X.index], saved.loc[X.index, f"{outcome}_label"]
        )
    print("PASS: recorded Experiment 1 patient membership and FA/GI labels match.")
else:
    print("No recorded split: baseline used the current Experiment 1 split function.")
print("PASS: unique patients, disjoint partitions, aligned FA/GI labels, clinical-only numeric X.")
print("No model was trained and no experiment result was overwritten.")